# E9: Integrated Pipeline — Inspection, Evaluation & Error Analysis

## Three modes

1. **Inspection** (Section 2): Run ONE game step-by-step, print every decision
2. **Evaluation** (Section 3): Run N games, collect metrics
3. **Error Analysis** (Section 4): Post-hoc study of failures

## Pipeline under test

```
RETRIEVE:  causal_search → k episodes with continuations
VALIDATE:  KG.filter_valid_actions → remove invalid actions
SCORE:     BN.query → P(success | task, phase, region, action)
ACT:       LLM chooses from scored + validated actions
LEARN:     outcome → typed_edges + BN.observe + WeightLearner
```

**Model**: Ollama qwen2.5:7b (zero cost)


In [1]:
import os, json, time, re
from sentence_transformers import SentenceTransformer
from openai import OpenAI

# === LLM via Ollama (OpenAI-compatible API) ===
OLLAMA_URL = 'http://172.16.244.34:11434/v1'
client = OpenAI(base_url=OLLAMA_URL, api_key='ollama')
LLM_MODEL = 'qwen2.5:14b'

# === Embedder ===
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Constants
TOP_K = 5
MAX_STEPS = 30
N_EVAL = 30

print(f'LLM: {LLM_MODEL}')
print(f'Embedder: {embedder.get_sentence_embedding_dimension()}D')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LLM: qwen2.5:14b
Embedder: 384D


## 1. Build Memory Components

Three components initialized from expert data:
- **CVX Index**: 336 expert episodes with temporal edges
- **Knowledge Graph**: task structure with preconditions/effects  
- **Bayesian Network**: P(success | task_type, phase, action)


In [2]:
import chronos_vector as cvx

# === CVX Index ===
with open('../data/episodic/alfworld_metadata.json') as f:
    meta = json.load(f)

index = cvx.TemporalIndex(m=16, ef_construction=100)
action_lookup, task_lookup = {}, {}

for ep_id_str, ep in meta.items():
    ep_id = int(ep_id_str)
    task = ep.get('task', '')
    task_lookup[ep_id] = task
    for si, step in enumerate(ep['steps']):
        text = f"{task} | {step.get('action','')} | {step.get('observation','')}"
        vec = embedder.encode(text).tolist()
        ts = ep_id * 10000 + si
        index.insert(ep_id, ts, vec, reward=1.0)
        action_lookup[ts] = step.get('action', '')

print(f'CVX Index: {len(index)} points from {len(meta)} episodes')
print(f'Action lookup: {len(action_lookup)} entries')


CVX Index: 4542 points from 336 episodes
Action lookup: 4542 entries


In [3]:
# === Task type detection ===
TASK_TYPES = {
    'heat': 'heat', 'cool': 'cool', 'clean': 'clean',
    'examine': 'examine', 'look at': 'examine',
    'find two': 'pick_two', 'put two': 'pick_two',
}

def detect_task_type(task_text):
    for kw, tt in TASK_TYPES.items():
        if kw in task_text.lower(): return tt
    return 'pick'

STRATEGIES = {
    'pick': 'Find the object, take it, go to target, put it.',
    'heat': 'Find object, take, go to microwave, heat, take, go to target, put.',
    'cool': 'Find object, take, go to fridge, cool, take, go to target, put.',
    'clean': 'Find object, take, go to sinkbasin, clean, go to target, put.',
    'examine': 'Find object, take, find lamp, use lamp.',
    'pick_two': 'Find first, take, put at target. Find second, take, put at target.',
}

# === Phase detection from FULL action history ===
def detect_phase(observation, history):
    """Infer phase from full action history (state machine).
    
    searching → holding → transforming → placing → searching (for pick_two)
    Scans ALL history to find last take/transform/put events.
    """
    if not history:
        return 'searching'

    all_actions = [h['action'].lower() for h in history]

    # Find indices of key events
    last_take = -1
    last_transform = -1  # heat/cool/clean (not navigation)
    last_put = -1
    last_use = -1  # for examine tasks

    for i, a in enumerate(all_actions):
        if a.startswith(('take', 'pick')):
            last_take = i
        elif a.startswith(('heat', 'cool', 'clean')) and not a.startswith(('go', 'close')):
            last_transform = i
        elif a.startswith('put'):
            last_put = i
        elif a.startswith('use'):
            last_use = i

    # 1. PUT after everything → back to searching (pick_two needs second object)
    if last_put > last_take and last_put > last_transform:
        return 'searching'

    # 2. TRANSFORMED after taking → placing (go to target + put)
    if last_transform > last_take and last_transform >= 0:
        return 'placing'

    # 3. USED a lamp (examine task) → placing (effectively done)
    if last_use > last_take and last_use >= 0:
        return 'placing'

    # 4. TOOK something → holding or transforming
    if last_take >= 0 and last_take > last_put:
        obs_lower = observation.lower()
        if any(x in obs_lower for x in ['microwave', 'fridge', 'sinkbasin']):
            return 'transforming'
        return 'holding'

    # 5. Default
    return 'searching'

# === Action type classification ===
def classify_action(action_text):
    a = action_text.lower()
    if a.startswith('go to'): return 'navigate'
    if a.startswith('take') or a.startswith('pick'): return 'take'
    if a.startswith('put'): return 'place'
    if a.startswith('open'): return 'open'
    if a.startswith('close'): return 'close'
    if a.startswith('clean'): return 'clean'
    if a.startswith('heat') or a.startswith('cook'): return 'heat'
    if a.startswith('cool'): return 'cool'
    if a.startswith('use'): return 'use'
    if a.startswith('examine') or a.startswith('look'): return 'examine'
    return 'other'

print('Phase detection + action classification ready')
print(f'Task types: {list(STRATEGIES.keys())}')


Phase detection + action classification ready
Task types: ['pick', 'heat', 'cool', 'clean', 'examine', 'pick_two']


In [4]:
from alfworld.agents.environment.alfred_tw_env import AlfredTWEnv

config = {
    'dataset': {
        'data_path': os.path.expanduser('~/.cache/alfworld/json_2.1.1/train'),
        'eval_id_data_path': os.path.expanduser('~/.cache/alfworld/json_2.1.1/valid_seen'),
        'eval_ood_data_path': os.path.expanduser('~/.cache/alfworld/json_2.1.1/valid_unseen'),
        'num_train_games': 0, 'num_eval_games': 0,
    },
    'env': {'goal_desc_human_anns_prob': 0.0, 'task_types': [1,2,3,4,5,6],
            'domain_randomization': False, 'expert_type': 'handcoded'},
    'general': {'training_method': 'dqn', 'random_seed': 42},
    'rl': {'training': {'max_nb_steps_per_episode': MAX_STEPS}},
    'dagger': {'training': {'max_nb_steps_per_episode': MAX_STEPS}},
    'logic': {'domain': os.path.expanduser('~/.cache/alfworld/logic/alfred.pddl'),
              'grammar': os.path.expanduser('~/.cache/alfworld/logic/alfred.twl2')},
}
env_wrapper = AlfredTWEnv(config, train_eval='eval_out_of_distribution')
env = env_wrapper.init_env(batch_size=1)
print(f'ALFWorld: {len(env_wrapper.game_files)} games')


Initializing AlfredTWEnv...


  0%|          | 0/341 [00:00<?, ?it/s]

 75%|███████▍  | 255/341 [00:00<00:00, 2549.23it/s]

100%|██████████| 341/341 [00:00<00:00, 1492.15it/s]

Overall we have 134 games in split=eval_out_of_distribution
Evaluating with 134 games
ALFWorld: 134 games


## 2. Pipeline Functions

Each step returns diagnostic information for inspection.


In [5]:
from collections import Counter
import numpy as np
import re

# === Build RegionMDP from expert episodes ===
from cvx_utils.region_mdp import RegionMDP, classify_alfworld_action

mdp = RegionMDP(index, level=1)

for ep_id_str, ep in meta.items():
    ep_id = int(ep_id_str)
    traj = index.trajectory(ep_id)
    if len(traj) < 2:
        continue
    regions = []
    actions = []
    for ts, vec in traj:
        region = index.assign_region(vec, 1)
        regions.append(region if region is not None else -1)
        action_text = action_lookup.get(ts, 'other')
        actions.append(classify_alfworld_action(action_text))
    mdp.learn_from_trajectory(regions, actions, 1.0)

print(f'RegionMDP: {mdp.stats()}')

# === Location hints extracted from expert data ===
LOCATION_HINTS = {
    ('cool', 'searching'): 'Search countertops, cabinets, and drawers for the object.',
    ('cool', 'holding'):   'Go to the FRIDGE to cool the object.',
    ('cool', 'transforming'): 'Cool the object in the fridge, then take it back.',
    ('cool', 'placing'):   'Go to the target location (check the task) and put the object.',
    ('heat', 'searching'): 'Search cabinets, countertops, and dining table for the object.',
    ('heat', 'holding'):   'Go to the MICROWAVE to heat the object.',
    ('heat', 'transforming'): 'Heat/cook the object in the microwave, then take it back.',
    ('heat', 'placing'):   'Go to the target location and put the object.',
    ('clean', 'searching'): 'Search cabinets, drawers, sinkbasin, and fridge for the object.',
    ('clean', 'holding'):   'Go to the SINKBASIN to clean the object.',
    ('clean', 'transforming'): 'Clean the object with the sinkbasin.',
    ('clean', 'placing'):   'Go to the target location (countertop, cabinet, etc.) and put the object.',
    ('examine', 'searching'): 'Search desks, drawers, beds, shelves, sidetables for the object.',
    ('examine', 'holding'):   'Find a DESKLAMP or FLOORLAMP and USE it to examine the object.',
    ('examine', 'transforming'): 'USE the lamp to examine the object.',
    ('examine', 'placing'):   'Task complete after using the lamp.',
    ('pick', 'searching'): 'Search drawers, cabinets, countertops, shelves, tables for the object.',
    ('pick', 'holding'):   'Go to the TARGET location mentioned in the task and put the object.',
    ('pick', 'transforming'): 'Go to the target and put the object.',
    ('pick', 'placing'):   'Put the object at the target location.',
    ('pick_two', 'searching'): 'Search drawers, shelves, cabinets, desks, countertops for the objects.',
    ('pick_two', 'holding'):   'Go to the TARGET location and put the object. Then find the second one.',
    ('pick_two', 'transforming'): 'Put the object at the target, then search for the next one.',
    ('pick_two', 'placing'):   'Put the object at the target location.',
}

def extract_task_target(task_text):
    t = task_text.lower()
    m = re.search(r'(?:put|place)\s+.+?\s+(?:in|on|into)\s+(\w+)', t)
    if m:
        return m.group(1)
    return None

def extract_task_object(task_text):
    t = task_text.lower()
    for pattern in [
        r'(?:put|place)\s+(?:a|some|the|two)?\s*(?:hot|cool|clean|heated|cooled|cleaned)?\s*(\w+)',
        r'(?:heat|cool|clean)\s+(?:a|some|the)?\s*(\w+)',
        r'(?:examine|look at)\s+(?:a|some|the)?\s*(\w+)',
        r'(?:find)\s+(?:a|some|the|two)?\s*(\w+)',
    ]:
        m = re.search(pattern, t)
        if m:
            obj = m.group(1)
            if obj not in ('a', 'some', 'the', 'two', 'hot', 'cool', 'clean'):
                return obj
    return None


def parse_world_state(history, current_obs=''):
    """Build world state from action+observation history."""
    seen_objects = {}  # object → location where last seen
    holding = None
    holding_state = None  # 'cooled', 'heated', 'cleaned'
    current_location = None

    all_obs = [(h['action'].lower(), h['obs'].lower()) for h in (history or [])]

    for action, obs in all_obs:
        # Track current location
        loc_m = re.search(r'arrive at (\w+ \d+)', obs)
        if loc_m:
            current_location = loc_m.group(1)

        # Parse objects from "you see X, Y, and Z" patterns
        see_m = re.search(r'you see (.+?)\.?$', obs)
        if see_m and current_location:
            objects_text = see_m.group(1)
            for obj_m in re.finditer(r'(?:a |the )?(\w+ \d+)', objects_text):
                obj = obj_m.group(1)
                if obj != current_location:  # don't count location as object
                    seen_objects[obj] = current_location

        # Track inventory
        if action.startswith(('take', 'pick')):
            m = re.match(r'(?:take|pick up) (\w+ \d+)', action)
            if m:
                holding = m.group(1)
                holding_state = None
        if action.startswith('cool'):
            holding_state = 'cooled'
        elif action.startswith('heat') and not action.startswith('go'):
            holding_state = 'heated'
        elif action.startswith('clean') and not action.startswith('go'):
            holding_state = 'cleaned'
        if action.startswith(('put', 'move')):
            holding = None
            holding_state = None

    return holding, holding_state, seen_objects


def format_world_state(holding, holding_state, seen_objects, target_obj):
    """Format world state — holding + target location."""
    parts = []

    # Inventory
    if holding:
        state_str = f' ({holding_state})' if holding_state else ''
        parts.append(f'Holding: {holding}{state_str}')
    
    # Target object location
    if target_obj and not holding:
        target_locations = [(obj, loc) for obj, loc in seen_objects.items() 
                           if target_obj in obj]
        if target_locations:
            obj, loc = target_locations[0]
            parts.append(f'TARGET FOUND: {obj} is at {loc} — go there and take it!')

    return '\n'.join(parts)


def get_abstract_guidance(task_type, phase, current_region, mdp, admissible, task_text, history=None):
    parts = []

    target_obj = extract_task_object(task_text)

    strategy = STRATEGIES.get(task_type, '')
    if strategy:
        parts.append(f'Strategy: {strategy}')
    hint = LOCATION_HINTS.get((task_type, phase), '')
    if hint:
        parts.append(f'NOW: {hint}')
    target_loc = extract_task_target(task_text)
    if target_obj and phase == 'searching':
        parts.append(f'Looking for: {target_obj}. ONLY take {target_obj}, ignore other objects.')
    if target_loc and phase in ('placing', 'holding') and task_type in ('pick', 'pick_two'):
        parts.append(f'Target destination: {target_loc}')
    if current_region is not None and current_region != -1:
        best = mdp.best_actions(current_region)
        if best:
            parts.append(f'Best action type here: {best[0][0]} ({best[0][1]:.0%} success)')

    # FIX: In searching phase, show unvisited locations
    if phase == 'searching' and admissible:
        nav_actions = [a for a in admissible if a.lower().startswith('go to')]
        visited_locs = set()
        for h in (history or []):
            a = h['action'].lower()
            if a.startswith('go to'):
                visited_locs.add(a)
        unvisited = [a for a in nav_actions if a.lower() not in visited_locs]
        if unvisited and len(history or []) >= 3:
            parts.append(f'NOT YET VISITED: {", ".join(unvisited[:8])}')

    # FIX: In holding phase for pick/pick_two, show PUT directive
    if phase == 'holding' and task_type in ('pick', 'pick_two'):
        target_loc = extract_task_target(task_text)
        if target_loc:
            # Find admissible "go to <target>" actions
            target_navs = [a for a in admissible if target_loc in a.lower() and a.lower().startswith('go to')]
            if target_navs:
                parts.append(f'GO TO TARGET NOW: {target_navs[0]}')
            # If already at target, show put action
            put_actions = [a for a in admissible if a.lower().startswith('put')]
            if put_actions:
                parts.append(f'USE THIS ACTION NOW: {put_actions[0]}')

    # FIX: In placing phase, explicitly tell LLM to use PUT
    if phase == 'placing':
        put_actions = [a for a in admissible if a.lower().startswith('put')]
        if put_actions:
            parts.append(f'USE THIS ACTION NOW: {put_actions[0]}')
        else:
            # Need to open container first, or navigate to target
            target_loc = extract_task_target(task_text)
            if target_loc:
                parts.append(f'Go to {target_loc} and OPEN it, then PUT the object.')

    # FIX: In transforming phase, be explicit about the action verb
    if phase == 'transforming':
        task_type_local = detect_task_type(task_text)
        if task_type_local == 'clean':
            clean_actions = [a for a in admissible if a.lower().startswith('clean')]
            if clean_actions:
                parts.append(f'USE THIS ACTION NOW: {clean_actions[0]}')
        elif task_type_local == 'heat':
            heat_actions = [a for a in admissible if a.lower().startswith('heat')]
            if heat_actions:
                parts.append(f'USE THIS ACTION NOW: {heat_actions[0]}')
        elif task_type_local == 'cool':
            cool_actions = [a for a in admissible if a.lower().startswith('cool')]
            if cool_actions:
                parts.append(f'USE THIS ACTION NOW: {cool_actions[0]}')

    return '\n'.join(parts) + '\n'

def detect_loops(history, threshold=2):
    if len(history) < threshold + 1:
        return set()
    exclude = set()
    recent = [h['action'] for h in history[-(threshold+1):]]
    if len(set(recent[-threshold:])) == 1:
        exclude.add(recent[-1])
    if len(history) >= 4:
        last4 = [h['action'] for h in history[-4:]]
        if last4[0] == last4[2] and last4[1] == last4[3]:
            exclude.add(last4[-1])
            exclude.add(last4[-2])
    if len(history) >= 6:
        last6 = [h['action'] for h in history[-6:]]
        if last6[0] == last6[3] and last6[1] == last6[4] and last6[2] == last6[5]:
            exclude.update(last6[-3:])

    # FIX: Track locations visited with nothing useful found
    # After 10+ steps of searching, exclude locations we already checked
    if len(history) >= 10:
        visited_empty = set()
        for h in history:
            a = h['action'].lower()
            obs = h['obs'].lower()
            if a.startswith('go to'):
                if 'nothing' in obs or ('closed' in obs and 'open' not in obs):
                    visited_empty.add(a)
                else:
                    # Found something here — keep it available
                    visited_empty.discard(a)
        # Only exclude if we have alternatives
        nav_actions = [a for a in exclude if a.startswith('go to')]
        remaining_nav = len([h['action'] for h in history if h['action'].startswith('go to')]) 
        if len(visited_empty) < 20:  # don't exclude everything
            exclude.update(visited_empty)

    return exclude

# === Few-shot examples (1 per task type) ===
FEW_SHOT = {
    'pick': """Example — Task: put some newspaper on garbagecan.
> go to coffeetable 1 (search for object)
> take newspaper 2 from coffeetable 1 (take it)
> go to garbagecan 1 (go to target)
> put newspaper 2 in/on garbagecan 1 (place it — DONE)""",

    'heat': """Example — Task: heat some mug and put it in coffeemachine.
> go to countertop 1 (search for mug)
> take mug 1 from countertop 1 (take it)
> go to microwave 1 (go to microwave)
> heat mug 1 with microwave 1 (heat it)
> go to coffeemachine 1 (go to target)
> put mug 1 in/on coffeemachine 1 (place it — DONE)""",

    'cool': """Example — Task: put a cool cup in shelf.
> go to countertop 1 (search for cup)
> take cup 2 from countertop 1 (take it)
> go to fridge 1 (go to fridge)
> cool cup 2 with fridge 1 (cool it)
> go to shelf 1 (go to target)
> put cup 2 in/on shelf 1 (place it — DONE)""",

    'clean': """Example — Task: put a clean soapbar in toilet.
> go to toilet 1 (search for soapbar)
> take soapbar 4 from toilet 1 (take it)
> go to sinkbasin 1 (go to sinkbasin)
> clean soapbar 4 with sinkbasin 1 (clean it)
> go to toilet 1 (go to target)
> put soapbar 4 in/on toilet 1 (place it — DONE)""",

    'examine': """Example — Task: examine the cellphone with the desklamp.
> go to desk 1 (search for cellphone)
> take cellphone 4 from desk 1 (take it)
> use desklamp 1 (use lamp to examine — DONE)""",

    'pick_two': """Example — Task: put two keychain in ottoman.
> go to diningtable 1 (search for first)
> take keychain 2 from diningtable 1 (take first)
> go to ottoman 1 (go to target)
> put keychain 2 in/on ottoman 1 (place first)
> go to sidetable 1 (search for second)
> take keychain 1 from sidetable 1 (take second)
> go to ottoman 1 (go to target again)
> put keychain 1 in/on ottoman 1 (place second — DONE)""",
}

def llm_decide(observation, admissible, task, history, context, verbose=False):
    """LLM chooses action with few-shot example + guidance + loop avoidance."""
    excluded = detect_loops(history)
    filtered = [a for a in admissible if a not in excluded]
    if not filtered:
        filtered = admissible

    if excluded and verbose:
        print(f'  LOOP DETECTED: excluding {excluded}')

    system = ('You are a household agent. You MUST choose EXACTLY ONE action '
              'from the Available actions list. Copy it EXACTLY as written. '
              'IMPORTANT: use "in/on" when placing objects (e.g. put X in/on Y).')

    user = f'Task: {task}\n\n'
    if context:
        user += context + '\n'
    if history:
        user += 'Recent:\n'
        for h in history[-3:]:
            user += f'  > {h["action"]}\n'
        user += '\n'
    user += f'Obs: {observation}\n\nAvailable actions:\n'
    for a in filtered:
        user += f'  - {a}\n'
    user += '\nChoose ONE action (copy exactly from the list above):'

    if verbose:
        print(f'  CONTEXT TO LLM:')
        for line in context.split('\n'):
            if line.strip():
                print(f'    {line}')

    try:
        resp = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{'role':'system','content':system},{'role':'user','content':user}],
            temperature=0.0, max_tokens=100,
        )
        chosen = resp.choices[0].message.content.strip().lower()
    except:
        chosen = filtered[0].lower()

    # Normalize ALFWorld "in/on" format for matching
    def normalize(s):
        return s.replace(' in/on ', ' in ').replace(' in/on ', ' on ')

    chosen_norm = normalize(chosen)

    for a in filtered:
        if a.lower() == chosen:
            if verbose: print(f'  LLM: exact "{a}"')
            return a
    for a in filtered:
        if normalize(a.lower()) == chosen_norm:
            if verbose: print(f'  LLM: exact (normalized) "{a}"')
            return a
    for a in filtered:
        a_norm = normalize(a.lower())
        if a_norm in chosen_norm or chosen_norm in a_norm:
            if verbose: print(f'  LLM: partial "{a}"')
            return a
    if verbose: print(f'  LLM: fallback (raw: "{chosen[:40]}")')
    return filtered[0]

def online_learn(index, mdp, history, task, won):
    next_ep_id = max(int(k) for k in meta.keys()) + 1 + len(online_episodes)
    online_episodes.append(next_ep_id)
    reward = 1.0 if won else 0.0
    regions = []
    actions = []
    for si, h in enumerate(history):
        text = f"{task} | {h['action']} | {h['obs']}"
        vec = embedder.encode(text).tolist()
        ts = next_ep_id * 10000 + si
        if won:
            index.insert(next_ep_id, ts, vec, reward=reward)
            action_lookup[ts] = h['action']
        region = index.assign_region(vec, 1)
        regions.append(region if region is not None else -1)
        actions.append(classify_alfworld_action(h['action']))
    mdp.learn_from_trajectory(regions, actions, reward)

online_episodes = []

print('\nPipeline ready (few-shot + location hints + loop detection + online learning)')


RegionMDP: 4206 transitions, 268 states, 11 action types

Pipeline ready (few-shot + location hints + loop detection + online learning)


## 2. Inspection Mode

Run ONE game step-by-step. See every decision the pipeline makes.


In [6]:
def run_game_inspect(env):
    """Run one game with diagnostic output."""
    obs, info = env.reset()
    observation = obs[0]
    task = observation.split('Your task is to:')[-1].strip().split('\n')[0] if 'Your task is to:' in observation else ''
    task_type = detect_task_type(task)

    print(f'═══ GAME: {task} ═══')
    print(f'Task type: {task_type}')
    print(f'Strategy: {STRATEGIES.get(task_type, "?")}')
    print(f'Target object: {extract_task_object(task)}')
    print(f'Target location: {extract_task_target(task)}')
    print()

    history = []

    for step in range(MAX_STEPS):
        admissible = info['admissible_commands'][0]
        phase = detect_phase(observation, history)

        query_vec = embedder.encode(f'{task} | {observation}').tolist()
        current_region = index.assign_region(query_vec, 1)

        print(f'── Step {step+1} ── phase={phase}, region={current_region}')
        print(f'  OBS: {observation[:120]}')

        context = get_abstract_guidance(task_type, phase, current_region, mdp, admissible, task, history)
        action = llm_decide(observation, admissible, task, history, context, verbose=True)

        obs, rewards, dones, info = env.step([action])
        observation = obs[0]
        history.append({'action': action, 'obs': observation[:100]})

        print(f'  ACTION: {action}')
        print(f'  RESULT: {observation[:80]}')
        print()

        if dones[0]: break

    won = info['won'][0]
    print(f'{"WIN" if won else "FAIL"} in {len(history)} steps')
    online_learn(index, mdp, history, task, won)
    print(f'  (online: {"added to index + " if won else ""}updated RegionMDP)')

    return {'task': task, 'task_type': task_type, 'won': won,
            'steps': len(history), 'history': history}

result = run_game_inspect(env)


═══ GAME: put a cool tomato in microwave. ═══
Task type: cool
Strategy: Find object, take, go to fridge, cool, take, go to target, put.
Target object: tomato
Target location: microwave

── Step 1 ── phase=searching, region=1224
  OBS: -= Welcome to TextWorld, ALFRED! =-

You are in the middle of a room. Looking quickly around you, you see a cabinet 6, a
  CONTEXT TO LLM:
    Strategy: Find object, take, go to fridge, cool, take, go to target, put.
    NOW: Search countertops, cabinets, and drawers for the object.
    Looking for: tomato. ONLY take tomato, ignore other objects.
    Best action type here: navigate (92% success)


  LLM: exact "go to countertop 1"
  ACTION: go to countertop 1
  RESULT: You arrive at countertop 1. On the countertop 1, you see a dishsponge 1, and a m

── Step 2 ── phase=searching, region=1224
  OBS: You arrive at countertop 1. On the countertop 1, you see a dishsponge 1, and a mug 2.
  CONTEXT TO LLM:
    Strategy: Find object, take, go to fridge, cool, take, go to target, put.
    NOW: Search countertops, cabinets, and drawers for the object.
    Looking for: tomato. ONLY take tomato, ignore other objects.
    Best action type here: navigate (92% success)


  LLM: exact "go to cabinet 1"
  ACTION: go to cabinet 1
  RESULT: You arrive at cabinet 1. On the cabinet 1, you see nothing.

── Step 3 ── phase=searching, region=3268
  OBS: You arrive at cabinet 1. On the cabinet 1, you see nothing.
  CONTEXT TO LLM:
    Strategy: Find object, take, go to fridge, cool, take, go to target, put.
    NOW: Search countertops, cabinets, and drawers for the object.
    Looking for: tomato. ONLY take tomato, ignore other objects.
    Best action type here: navigate (89% success)


  LLM: exact "go to cabinet 2"
  ACTION: go to cabinet 2
  RESULT: You arrive at cabinet 2. The cabinet 2 is closed.

── Step 4 ── phase=searching, region=4023
  OBS: You arrive at cabinet 2. The cabinet 2 is closed.
  CONTEXT TO LLM:
    Strategy: Find object, take, go to fridge, cool, take, go to target, put.
    NOW: Search countertops, cabinets, and drawers for the object.
    Looking for: tomato. ONLY take tomato, ignore other objects.
    Best action type here: open (90% success)
    NOT YET VISITED: go to cabinet 3, go to cabinet 4, go to cabinet 5, go to cabinet 6, go to coffeemachine 1, go to countertop 2, go to countertop 3, go to drawer 1


  LLM: exact "open cabinet 2"
  ACTION: open cabinet 2
  RESULT: You open the cabinet 2. The cabinet 2 is open. In it, you see a plate 1.

── Step 5 ── phase=searching, region=749
  OBS: You open the cabinet 2. The cabinet 2 is open. In it, you see a plate 1.
  CONTEXT TO LLM:
    Strategy: Find object, take, go to fridge, cool, take, go to target, put.
    NOW: Search countertops, cabinets, and drawers for the object.
    Looking for: tomato. ONLY take tomato, ignore other objects.
    Best action type here: navigate (96% success)
    NOT YET VISITED: go to cabinet 3, go to cabinet 4, go to cabinet 5, go to cabinet 6, go to coffeemachine 1, go to countertop 2, go to countertop 3, go to drawer 1


...

  LLM: exact "open fridge 1"
  ACTION: open fridge 1
  RESULT: You open the fridge 1. The fridge 1 is open. In it, you see a bread 1, a mug 3, 

── Step 11 ── phase=transforming, region=1229
  OBS: You open the fridge 1. The fridge 1 is open. In it, you see a bread 1, a mug 3, a pan 1, and a plate 3.
  LOOP DETECTED: excluding {'go to fridge 1', 'go to cabinet 2', 'go to cabinet 1'}
  CONTEXT TO LLM:
    Strategy: Find object, take, go to fridge, cool, take, go to target, put.
    NOW: Cool the object in the fridge, then take it back.
    Best action type here: navigate (90% success)
    USE THIS ACTION NOW: cool tomato 1 with fridge 1


  LLM: exact "cool tomato 1 with fridge 1"
  ACTION: cool tomato 1 with fridge 1
  RESULT: You cool the tomato 1 using the fridge 1.

── Step 12 ── phase=placing, region=1229
  OBS: You cool the tomato 1 using the fridge 1.
  LOOP DETECTED: excluding {'go to fridge 1', 'go to cabinet 2', 'go to cabinet 1'}
  CONTEXT TO LLM:
    Strategy: Find object, take, go to fridge, cool, take, go to target, put.
    NOW: Go to the target location (check the task) and put the object.
    Best action type here: navigate (90% success)
    Go to microwave and OPEN it, then PUT the object.


  LLM: exact "go to microwave 1"
  ACTION: go to microwave 1
  RESULT: You arrive at microwave 1. The microwave 1 is closed.

── Step 13 ── phase=placing, region=1229
  OBS: You arrive at microwave 1. The microwave 1 is closed.
  LOOP DETECTED: excluding {'go to fridge 1', 'go to microwave 1', 'go to cabinet 2', 'go to cabinet 1'}
  CONTEXT TO LLM:
    Strategy: Find object, take, go to fridge, cool, take, go to target, put.
    NOW: Go to the target location (check the task) and put the object.
    Best action type here: navigate (90% success)
    Go to microwave and OPEN it, then PUT the object.


  LLM: exact "open microwave 1"
  ACTION: open microwave 1
  RESULT: You open the microwave 1. The microwave 1 is open. In it, you see a egg 1.

── Step 14 ── phase=placing, region=1229
  OBS: You open the microwave 1. The microwave 1 is open. In it, you see a egg 1.
  LOOP DETECTED: excluding {'go to fridge 1', 'go to microwave 1', 'go to cabinet 2', 'go to cabinet 1'}
  CONTEXT TO LLM:
    Strategy: Find object, take, go to fridge, cool, take, go to target, put.
    NOW: Go to the target location (check the task) and put the object.
    Best action type here: navigate (90% success)
    Go to microwave and OPEN it, then PUT the object.


  LLM: exact "move tomato 1 to microwave 1"
  ACTION: move tomato 1 to microwave 1
  RESULT: You move the tomato 1 to the microwave 1.

WIN in 14 steps
  (online: added to index + updated RegionMDP)


## 3. Evaluation Mode

Run N games, collect aggregate metrics.


In [7]:
def run_game_eval(env):
    """Run one game silently."""
    obs, info = env.reset()
    observation = obs[0]
    task = observation.split('Your task is to:')[-1].strip().split('\n')[0] if 'Your task is to:' in observation else ''
    task_type = detect_task_type(task)
    history = []
    phases_seen = []
    action_types_used = []

    for step in range(MAX_STEPS):
        admissible = info['admissible_commands'][0]
        phase = detect_phase(observation, history)
        phases_seen.append(phase)

        query_vec = embedder.encode(f'{task} | {observation}').tolist()
        current_region = index.assign_region(query_vec, 1)

        context = get_abstract_guidance(task_type, phase, current_region, mdp, admissible, task, history)
        action = llm_decide(observation, admissible, task, history, context)

        action_types_used.append(classify_alfworld_action(action))
        obs, rewards, dones, info = env.step([action])
        observation = obs[0]
        history.append({'action': action, 'obs': observation[:100]})
        if dones[0]: break

    won = info['won'][0]
    online_learn(index, mdp, history, task, won)

    return {
        'task': task, 'task_type': task_type, 'won': won,
        'steps': len(history), 'history': history,
        'phases': phases_seen, 'action_types': action_types_used,
    }

print(f'Running {N_EVAL} games with location hints + loop detection + online learning...')
all_results = []
wins = 0
for g in range(N_EVAL):
    r = run_game_eval(env)
    all_results.append(r)
    if r['won']: wins += 1
    n_online = len(online_episodes)
    print(f'  {g+1}/{N_EVAL}: {"WIN" if r["won"] else "fail"} ({r["steps"]}s) {r["task_type"]} '
          f'[{wins}/{g+1}={wins/(g+1)*100:.0f}%] index={len(index)} online={n_online}')

print(f'\nResult: {wins}/{N_EVAL} = {wins/N_EVAL*100:.1f}%')
print(f'Online episodes added: {len(online_episodes)} ({sum(1 for r in all_results if r["won"])} wins)')
print(f'RegionMDP final: {mdp.stats()}')


Running 30 games with location hints + loop detection + online learning...


  1/30: fail (30s) clean [0/1=0%] index=4556 online=2


  2/30: WIN (6s) clean [1/2=50%] index=4562 online=3


  3/30: WIN (6s) pick [2/3=67%] index=4568 online=4


  4/30: WIN (11s) clean [3/4=75%] index=4579 online=5


...

  26/30: fail (30s) pick [17/26=65%] index=4726 online=27


  27/30: fail (30s) pick [17/27=63%] index=4726 online=28


  28/30: fail (30s) pick [17/28=61%] index=4726 online=29


  29/30: fail (30s) clean [17/29=59%] index=4726 online=30


  30/30: WIN (6s) cool [18/30=60%] index=4732 online=31

Result: 18/30 = 60.0%
Online episodes added: 31 (18 wins)
RegionMDP final: 4725 transitions, 280 states, 11 action types


In [8]:
from collections import defaultdict
# Per task type breakdown
print('\n=== Per Task Type ===')
by_type = defaultdict(list)
for r in all_results:
    by_type[r['task_type']].append(r['won'])

for tt in sorted(by_type):
    outcomes = by_type[tt]
    w = sum(outcomes)
    print(f'  {tt}: {w}/{len(outcomes)} = {w/len(outcomes)*100:.0f}%')

# Per phase distribution
print('\n=== Phase Distribution ===')
phase_counts = Counter()
for r in all_results:
    phase_counts.update(r['phases'])
total_phases = sum(phase_counts.values())
for phase, count in phase_counts.most_common():
    print(f'  {phase}: {count} ({count/total_phases*100:.0f}%)')

# Action type distribution in wins vs fails
print('\n=== Action Types: Wins vs Fails ===')
win_actions = Counter()
fail_actions = Counter()
for r in all_results:
    c = win_actions if r['won'] else fail_actions
    c.update(r['action_types'])
print('  Wins:', dict(win_actions.most_common(5)))
print('  Fails:', dict(fail_actions.most_common(5)))



=== Per Task Type ===
  clean: 3/5 = 60%
  cool: 4/6 = 67%
  examine: 4/4 = 100%
  heat: 2/4 = 50%
  pick: 4/10 = 40%
  pick_two: 1/1 = 100%

=== Phase Distribution ===
  searching: 272 (51%)
  holding: 161 (30%)
  placing: 69 (13%)
  transforming: 34 (6%)

=== Action Types: Wins vs Fails ===
  Wins: {'navigate': 87, 'open': 27, 'take': 19, 'other': 16, 'examine': 10}
  Fails: {'navigate': 120, 'examine': 103, 'open': 49, 'take': 32, 'other': 27}


## 4. Error Analysis

Post-hoc study of failures: why did the agent fail?


In [9]:
# === Error Analysis ===
failures = [r for r in all_results if not r['won']]
print(f'{len(failures)} failures to analyze')

# Failure patterns
print('\n=== Failure Patterns ===')

# 1. How many steps before failing?
step_counts = [r['steps'] for r in failures]
print(f'  Mean steps before failure: {np.mean(step_counts):.1f} (max={MAX_STEPS})')
print(f'  Timeout failures (hit MAX_STEPS): {sum(1 for s in step_counts if s >= MAX_STEPS)}')

# 2. What phase were they in when they failed?
print('\n=== Final Phase at Failure ===')
final_phases = Counter(r['phases'][-1] if r['phases'] else '?' for r in failures)
for phase, count in final_phases.most_common():
    print(f'  {phase}: {count}')

# 3. What task types fail most?
print('\n=== Failure Rate by Task Type ===')
for tt in sorted(by_type):
    outcomes = by_type[tt]
    fail_rate = 1 - sum(outcomes)/len(outcomes)
    print(f'  {tt}: {fail_rate*100:.0f}% failure rate ({len(outcomes)} games)')

# 4. Repetitive actions (agent stuck in loop)
print('\n=== Stuck in Loop? (same action 3+ times in a row) ===')
loop_count = 0
for r in failures:
    actions = [h['action'] for h in r['history']]
    for i in range(len(actions)-2):
        if actions[i] == actions[i+1] == actions[i+2]:
            loop_count += 1
            print(f'  Game "{r["task"][:30]}": repeated "{actions[i]}" at step {i+1}')
            break
print(f'  Total games with loops: {loop_count}/{len(failures)}')

# 5. Did retrieval help at all? Compare steps where candidates existed
print('\n=== Retrieval Quality ===')
# TODO: add retrieval metadata to step_log for deeper analysis
print('  (Enable inspection mode for per-step retrieval analysis)')


12 failures to analyze

=== Failure Patterns ===
  Mean steps before failure: 30.0 (max=30)
  Timeout failures (hit MAX_STEPS): 12

=== Final Phase at Failure ===
  holding: 6
  placing: 3
  searching: 3

=== Failure Rate by Task Type ===
  clean: 40% failure rate (5 games)
  cool: 33% failure rate (6 games)
  examine: 0% failure rate (4 games)
  heat: 50% failure rate (4 games)
  pick: 60% failure rate (10 games)
  pick_two: 0% failure rate (1 games)

=== Stuck in Loop? (same action 3+ times in a row) ===
  Total games with loops: 0/12

=== Retrieval Quality ===
  (Enable inspection mode for per-step retrieval analysis)


In [10]:
# === Inspect worst failures ===
print('=== Worst Failures (full action history) ===')
# Sort failures by step count (shortest = worst, couldn\'t even start)
worst = sorted(failures, key=lambda r: r['steps'])[:3]

for r in worst:
    print(f'\nTask: {r["task"]}')
    print(f'Type: {r["task_type"]}, Steps: {r["steps"]}')
    for i, h in enumerate(r['history'][:10]):
        print(f'  {i+1}. {h["action"]} → {h["obs"][:60]}')
    if len(r['history']) > 10:
        print(f'  ... ({len(r["history"])-10} more steps)')
    print()


=== Worst Failures (full action history) ===

Task: put a clean spatula in drawer.
Type: clean, Steps: 30
  1. go to drawer 1 → You arrive at drawer 1. The drawer 1 is closed.
  2. open drawer 1 → You open the drawer 1. The drawer 1 is open. In it, you see 
  3. close drawer 1 → You close the drawer 1.
  4. go to cabinet 1 → You arrive at cabinet 1. On the cabinet 1, you see a glassbo
  5. go to cabinet 2 → You arrive at cabinet 2. The cabinet 2 is closed.
  6. open cabinet 2 → You open the cabinet 2. The cabinet 2 is open. In it, you se
  7. go to drawer 1 → You arrive at drawer 1. The drawer 1 is closed.
  8. open drawer 1 → You open the drawer 1. The drawer 1 is open. In it, you see 
  9. go to cabinet 3 → You arrive at cabinet 3. On the cabinet 3, you see a vase 1.
  10. go to cabinet 4 → You arrive at cabinet 4. On the cabinet 4, you see nothing.
  ... (20 more steps)


Task: heat some egg and put it in garbagecan.
Type: heat, Steps: 30
  1. go to fridge 1 → You arrive at fridge 1

In [11]:
# Save for comparison across runs
results_data = {
    'config': {'model': LLM_MODEL, 'n_eval': N_EVAL, 'top_k': TOP_K},
    'summary': {'wins': wins, 'total': N_EVAL, 'rate': wins/N_EVAL},
    'per_task': {tt: {'wins': sum(o), 'total': len(o)} for tt, o in by_type.items()},
    'games': [{'task': r['task'], 'task_type': r['task_type'], 'won': r['won'], 
               'steps': r['steps']} for r in all_results],
}

with open('../data/episodic/e9_pipeline_results.json', 'w') as f:
    json.dump(results_data, f, indent=2)
print(f'Saved to data/episodic/e9_pipeline_results.json')


Saved to data/episodic/e9_pipeline_results.json
